# Parsli — backend playground

Same four-part flow as `compare_ollama_vs_lmstudio.ipynb`, but all logic comes
from the backend package instead of being re-implemented here.

| Part | What it does | Backend import |
|---|---|---|
| 1 | Gmail auth + email download | `parsli.gmail` |
| 2 | Deterministic preprocessing | `parsli.processing.cleaner`, `.rule_engine` |
| 3 | Model classification (LM Studio or llama-cpp) | `parsli.model` |
| 4 | Persistence + shipment resolution | `parsli.db`, `parsli.services` |

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "backend" / "src"))

import json
import time
from datetime import datetime, timezone
from typing import Literal

import pandas as pd

from parsli.config import AppConfig, GmailConfig, ModelConfig
from parsli.gmail.auth import GmailOAuthManager, TokenMissingError
from parsli.gmail.client import GmailClient
from parsli.gmail.candidate_fetcher import GmailCandidateFetcher
from parsli.gmail.query_builder import GmailQueryBuilder
from parsli.gmail.models import DomainPreferences
from parsli.gmail.sender_trust import SenderTrustScorer, SenderTrustLevel
from parsli.privacy.sanitizer import extract_sender_domain, clip_text
from parsli.privacy.hashing import sha256_hex

DATA_FILE = Path("data/emails.json")
DATA_FILE.parent.mkdir(exist_ok=True)

config = AppConfig(
    app_dir=PROJECT_ROOT / ".parsli",
    database={"sqlite_path": PROJECT_ROOT / ".parsli" / "parsli.db"},
)

print("project root :", PROJECT_ROOT)
print("credentials  :", config.credentials_path, "→", "✓" if config.credentials_path.exists() else "✗ NOT FOUND")
print("tokens dir   :", config.tokens_dir)
print("db           :", config.database.sqlite_path)
print("model        :", config.model.provider, "/", config.model.model_name)

In [ ]:
# ── DB setup — run once, available across all parts ───────────────────────────
from parsli.db.models import Base
from parsli.db.session import make_engine, make_session_factory

engine          = make_engine(config.database.sqlite_path)
Base.metadata.create_all(engine)
session_factory = make_session_factory(engine)

print(f"DB: {config.database.sqlite_path.resolve()}")

---
## Part 1 — Gmail auth + email download

In [ ]:
# List saved accounts so you can pick one
_oauth_list = GmailOAuthManager(
    credentials_path=config.credentials_path,
    tokens_dir=config.tokens_dir,
)
accounts = _oauth_list.list_token_accounts()

print("Saved accounts found:")
for a in accounts:
    print(f"  {a}")

ACCOUNT_ID = accounts[0] if accounts else ""
print(f"\nUsing:")

In [ ]:
oauth = GmailOAuthManager(
    credentials_path=config.credentials_path,
    tokens_dir=config.tokens_dir,
)

try:
    creds = oauth.refresh_if_needed(ACCOUNT_ID)
    print(f"Token loaded for: {ACCOUNT_ID}")
except TokenMissingError:
    print("No valid token — starting browser OAuth flow…")
    from google_auth_oauthlib.flow import InstalledAppFlow
    from googleapiclient.discovery import build as _gmail_build
    flow = InstalledAppFlow.from_client_secrets_file(
        str(config.credentials_path),
        scopes=GmailOAuthManager.SCOPES,
    )
    creds = flow.run_local_server(port=0)
    _profile = _gmail_build("gmail", "v1", credentials=creds).users().getProfile(userId="me").execute()
    ACCOUNT_ID = _profile["emailAddress"]
    oauth.save_token(ACCOUNT_ID, creds)
    print(f"Token saved for: {ACCOUNT_ID}")

In [ ]:
# ── Gmail query config — edit lookback_days or vocabulary here ───────────────
from parsli.services.domain_preference_service import DomainPreferenceService

gmail_cfg = GmailConfig(lookback_days=90)

with session_factory() as _s:
    domain_prefs = DomainPreferenceService(_s).get_preferences()

# Auto-add own email to exclude_senders if not already present
if ACCOUNT_ID and '@' in ACCOUNT_ID and ACCOUNT_ID not in domain_prefs.exclude_senders:
    with session_factory() as _s:
        DomainPreferenceService(_s).add_exclude_sender(ACCOUNT_ID)
        _s.commit()
    with session_factory() as _s:
        domain_prefs = DomainPreferenceService(_s).get_preferences()
    print(f"Auto-added {ACCOUNT_ID} to exclude_senders")

builder = GmailQueryBuilder(gmail_cfg, domain_prefs)
queries = builder.build_queries()

print(f"Allowlist       : {domain_prefs.allowlist or '(empty)'}")
print(f"Blocklist       : {domain_prefs.blocklist or '(empty)'}")
print(f"Exclude senders : {domain_prefs.exclude_senders or '(empty)'}")
print(f"\n{len(queries)} named queries:\n")
for q in queries:
    print(f"[{q.name}]")
    print(f"  {q.query}")
    print()

In [ ]:
# ── Manage domain/sender preferences ─────────────────────────────────────────
# Your own email is excluded automatically on the first sync — no action needed.
# Use this cell to add extra domains or senders, or to remove existing entries.
# Re-run the query config cell below after making any changes.

from parsli.services.domain_preference_service import DomainPreferenceService

# ── Edit these lists ──────────────────────────────────────────────────────────
ADD_EXCLUDE_SENDERS:    list[str] = []   # extra senders to exclude by email address
REMOVE_EXCLUDE_SENDERS: list[str] = []

ADD_BLOCKLIST:    list[str] = []         # whole domains to block from all queries
REMOVE_BLOCKLIST: list[str] = []

ADD_ALLOWLIST:    list[str] = []         # domains to get a dedicated allowlist query
REMOVE_ALLOWLIST: list[str] = []
# ─────────────────────────────────────────────────────────────────────────────

with session_factory() as _s:
    svc = DomainPreferenceService(_s)
    for s in ADD_EXCLUDE_SENDERS:    svc.add_exclude_sender(s)
    for s in REMOVE_EXCLUDE_SENDERS: svc.remove_exclude_sender(s)
    for d in ADD_BLOCKLIST:          svc.add_blocklist(d)
    for d in REMOVE_BLOCKLIST:       svc.remove_blocklist(d)
    for d in ADD_ALLOWLIST:          svc.add_allowlist(d)
    for d in REMOVE_ALLOWLIST:       svc.remove_allowlist(d)
    updated = svc.get_preferences()
    _s.commit()

print(f"Exclude senders : {updated.exclude_senders or '(set automatically on first sync)'}")
print(f"Blocklist       : {updated.blocklist or '(empty)'}")
print(f"Allowlist       : {updated.allowlist or '(empty)'}")

In [ ]:
client = GmailClient(creds)

fetcher    = GmailCandidateFetcher(client=client, builder=builder)
candidates = fetcher.fetch()

print(f"Unique candidates : {len(candidates.message_ids)}")
print(f"Matched >1 query  : {candidates.summary.multi_query_matches}")
print(f"Batch ID          : {candidates.summary.fetch_batch_id}")

# Keep a flat list for the download step below
message_ids = candidates.message_ids

In [ ]:
# ── Persist observability data ────────────────────────────────────────────────
# Saves query run timings and per-message source matches to the DB.
# Required for CandidateObservabilityService queries later in the session.
from parsli.services.candidate_observability_service import persist_fetch_result

with session_factory() as _s:
    persist_fetch_result(_s, candidates)
    _s.commit()

print("Query runs and candidate matches saved to DB.")

In [ ]:
# ── Query combination breakdown ───────────────────────────────────────────────
# candidates.summary is computed by the fetcher — no recomputation needed.
from collections import Counter

summary = candidates.summary

print("Per-query totals (use these with INSPECT_GROUP):")
print(f"  {'Query name':<32} {'Contributed':>11}  {'Exclusive':>9}")
print("  " + "─" * 57)

combo_counts = Counter(
    " + ".join(sorted(v)) for v in candidates.query_sources_by_message_id.values()
)
for name, total in sorted(summary.query_result_counts.items(), key=lambda x: -x[1]):
    exclusive = combo_counts.get(name, 0)
    print(f"  {name:<32} {total:>11}  {exclusive:>9}")

print()
print("Exact combination counts:")
print(f"  {'Combination':<50} {'Count':>6}")
print("  " + "─" * 58)
for combo, count in sorted(combo_counts.items(), key=lambda x: -x[1]):
    print(f"  {combo:<50} {count:>6}")
print("  " + "─" * 58)
print(f"  {'TOTAL':<50} {summary.total_unique_candidates:>6}")
print(f"  (batch: {summary.fetch_batch_id})")

In [ ]:
LIMIT = 300  # reduce for faster iteration

emails = []
for i, msg_id in enumerate(message_ids[:LIMIT]):
    raw = client.fetch_raw(msg_id)
    headers = client.extract_headers(raw)
    body = client.extract_body(raw.get('payload', {}))
    emails.append({
        'id':               msg_id,
        'subject':          headers.get('Subject', ''),
        'sender':           headers.get('From', ''),
        'date':             headers.get('Date', ''),
        'body':             body,
        'internal_date_ms': int(raw.get('internalDate', 0)),
    })
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{min(LIMIT, len(message_ids))}")

DATA_FILE.write_text(json.dumps(emails, ensure_ascii=False, indent=2))
print(f"\nSaved {len(emails)} emails → {DATA_FILE.resolve()}")

In [ ]:
# Inspect one email — change IDX to browse others
IDX = 0
e = emails[IDX]
print(f"Subject : {e['subject']}")
print(f"From    : {e['sender']}")
print(f"Date    : {e['date']}")
print(f"\nBody preview:\n{e['body'][:600]}")

In [ ]:
# ── Inspect emails by query group ────────────────────────────────────────────
# INSPECT_GROUP accepts:
#   - a single query name  →  "weak_phrases"   (all emails that matched it, including combos)
#   - an exact combo       →  "strong_shipping + weak_phrases"  (exact match only)
#
# SAVE_JSON = True  writes data/inspect_<group>.json for offline review.
# The JSON stores subject/sender/date/query_sources + a 300-char body preview
# (no full body stored to disk by default).

INSPECT_GROUP = "weak_phrases"   # ← edit this
SAVE_JSON     = True

# ── helpers ───────────────────────────────────────────────────────────────────
def _combo_key(sources: list[str]) -> str:
    return " + ".join(sorted(sources))

def _matches_group(sources: list[str], group: str) -> bool:
    if "+" in group:
        return _combo_key(sources) == group.strip()
    return group.strip() in sources

# ── filter ────────────────────────────────────────────────────────────────────
id_to_email  = {e["id"]: e for e in emails}
matched_ids  = [
    mid for mid, srcs in candidates.query_sources_by_message_id.items()
    if _matches_group(srcs, INSPECT_GROUP) and mid in id_to_email
]
matched      = [
    {**id_to_email[mid], "_sources": candidates.query_sources_by_message_id[mid]}
    for mid in matched_ids
]

print(f"Group '{INSPECT_GROUP}': {len(matched)} emails")

# ── save JSON ─────────────────────────────────────────────────────────────────
if SAVE_JSON and matched:
    slug     = INSPECT_GROUP.replace(" + ", "_").replace(" ", "_")
    out_path = Path(f"data/inspect_{slug}.json")
    payload  = [
        {
            "id":           e["id"],
            "subject":      e.get("subject", ""),
            "sender":       e.get("sender", ""),
            "date":         e.get("date", ""),
            "query_sources": e["_sources"],
            "body_preview": e.get("body", "")[:300],
        }
        for e in matched
    ]
    out_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2))
    print(f"Saved  → {out_path.resolve()}")

# ── display ───────────────────────────────────────────────────────────────────
pd.set_option("display.max_colwidth", 80)
pd.DataFrame([
    {
        "subject":      e.get("subject", "")[:70],
        "sender":       e.get("sender", "")[:45],
        "date":         e.get("date", "")[:16],
        "queries":      ", ".join(e["_sources"]),
        "body_preview": e.get("body", "")[:120].replace("\n", " "),
    }
    for e in matched
])

---
## Part 2 — Deterministic preprocessing

`EmailCleaner` strips HTML/boilerplate and returns a `CleanedEmail` DTO.  
`RuleEngine` applies regex + phrase rules and returns a `RuleExtractionResult` DTO.  
No model calls happen here.

In [ ]:
# Run this cell to reload from disk without re-downloading
emails = json.loads(DATA_FILE.read_text())
print(f"Loaded {len(emails)} emails from {DATA_FILE}")

In [ ]:
from parsli.processing.cleaner import EmailCleaner
from parsli.processing.rule_engine import RuleEngine

cleaner      = EmailCleaner()
rule_engine  = RuleEngine()
# Use the user's blocklist so blocked domains score as BLOCKED (not LOW)
_blocklist   = frozenset(getattr(globals().get('domain_prefs'), 'blocklist', []))
trust_scorer = SenderTrustScorer(user_blocklist=_blocklist)

preprocessed_rows = []
for e in emails:
    cleaned       = cleaner.clean(e['id'], e.get('body', ''))
    sender_domain = extract_sender_domain(e.get('sender', ''))
    rules         = rule_engine.extract(e['id'], cleaned.cleaned_text, sender_domain=sender_domain)
    trust         = trust_scorer.score(sender_domain)
    preprocessed_rows.append({
        'email_id':             e['id'],
        'subject':              e.get('subject', ''),
        'sender':               e.get('sender', ''),
        'date':                 e.get('date', ''),
        'internal_date_ms':     e.get('internal_date_ms', 0),
        'sender_domain':        sender_domain,
        # sender trust
        'trust_level':          trust.trust_level.value,
        'trust_score':          trust.trust_score,
        'trust_reasons':        trust.reasons,
        # cleaner outputs
        'cleaned_text':         cleaned.cleaned_text,
        'cleaned_full_len':     cleaned.cleaned_full_len,
        'cleaned_text_hash':    cleaned.cleaned_text_hash,
        'is_shipping_shaped':   cleaned.is_shipping_shaped,
        # rule outputs
        'rule_status':          rules.status.value if rules.status else None,
        'rule_confidence':      rules.status_confidence,
        'rule_evidence':        rules.status_evidence,
        'is_invoice':           rules.is_invoice,
        'is_shipping_email':    rules.is_shipping_email,
        'tracking_candidates':  [t.value for t in rules.tracking_candidates],
        'order_candidates':     [o.value for o in rules.order_candidates],
        'selected_tracking_number': rules.tracking_candidates[0].value if rules.tracking_candidates else None,
        'selected_order_number':    rules.order_candidates[0].value if rules.order_candidates else None,
        'merchant':             rules.merchant,
        'pickup_code':          rules.pickup_code,
        'amount':               rules.amount,
        'currency':             rules.currency,
    })

pre_df = pd.DataFrame(preprocessed_rows)

print(f"Preprocessed:     {len(pre_df)}")
print(f"Shipping-shaped:  {pre_df['is_shipping_shaped'].sum()}")
print(f"Rule-classified:  {pre_df['rule_status'].notna().sum()}")
print(f"Invoices/payment: {pre_df['is_invoice'].sum()}")
print(f"\nSender trust distribution:")
print(pre_df['trust_level'].value_counts().to_string())

In [ ]:
pre_df[[
    'subject', 'sender_domain',
    'trust_level', 'trust_score',
    'is_shipping_shaped', 'is_invoice',
    'rule_status', 'rule_confidence',
    'tracking_candidates', 'order_candidates',
]].head(20)

---
## Part 3 — Model classification

Rows the rule engine can't decide (`rule_status` is `None` or `unknown`) are
sent to the local model. Backend is switchable: `lmstudio` or `llamacpp`.

In [ ]:
from parsli.model.factory import ModelClientFactory
from parsli.model.base import ModelExtractionResult
from parsli.model.prompts import format_prompt

# ── Edit these ────────────────────────────────────────────────────────────────
BACKEND: Literal['lmstudio', 'llamacpp'] = 'lmstudio'

model_cfg = ModelConfig(
    provider=BACKEND,
    model_name='google/gemma-4-e4b',    # must match the model loaded in LM Studio
    endpoint_url='http://127.0.0.1:1234',
    timeout_seconds=120,
    max_input_chars=1500,
)
# ─────────────────────────────────────────────────────────────────────────────

model_client = ModelClientFactory.create(model_cfg)
print(f"Backend: {BACKEND}  model: {model_cfg.model_name}")

In [ ]:
# Smoke-test: one call before the full loop
_test = next((r for r in preprocessed_rows if r['is_shipping_shaped']), preprocessed_rows[0])
_prompt = format_prompt(clip_text(_test['cleaned_text'], model_cfg.max_input_chars), version='v1')
_t0 = time.perf_counter()
_res = model_client.extract(_prompt, response_model=ModelExtractionResult)
print(f"status={_res.status.value}  confidence={_res.status_confidence:.2f}  ({time.perf_counter()-_t0:.1f}s)")
print(f"evidence: {_res.status_evidence[:120]}")

In [ ]:
# Rule-first, model fallback on ambiguous rows
rows_out: list[dict] = []
rule_calls = model_calls = errors = 0

for i, pre in enumerate(preprocessed_rows, 1):
    row = dict(pre)
    rule_status  = pre['rule_status']
    rule_decided = rule_status is not None and rule_status != 'unknown'

    if rule_decided:
        row.update(
            status=rule_status,
            status_confidence=pre['rule_confidence'],
            status_evidence=pre['rule_evidence'],
            classification_method='rules',
            model_called=False,
            model_seconds=0.0,
        )
        rule_calls += 1
    else:
        clipped = clip_text(pre['cleaned_text'], model_cfg.max_input_chars)
        prompt  = format_prompt(clipped, version='v1')
        t0 = time.perf_counter()
        try:
            result  = model_client.extract(prompt, response_model=ModelExtractionResult)
            elapsed = round(time.perf_counter() - t0, 2)
            row.update(
                status=result.status.value,
                status_confidence=result.status_confidence,
                status_evidence=result.status_evidence,
                classification_method=BACKEND,
                model_called=True,
                model_seconds=elapsed,
            )
            # Model fills gaps the rules couldn't
            if not pre['selected_tracking_number'] and result.tracking_numbers:
                row['selected_tracking_number'] = result.tracking_numbers[0]
            if not pre['selected_order_number'] and result.order_numbers:
                row['selected_order_number'] = result.order_numbers[0]
            if not pre['merchant'] and result.merchant:
                row['merchant'] = result.merchant
            model_calls += 1
        except Exception as exc:
            print(f"  [{i}] error: {exc}")
            row.update(status='unknown', status_confidence=0.0, status_evidence='',
                       classification_method='error', model_called=True, model_seconds=0.0)
            errors += 1

    rows_out.append(row)

print(f"Done — rules={rule_calls}  model={model_calls}  errors={errors}")

In [ ]:
RESULTS_CSV = Path('data/results.csv')

df = pd.DataFrame(rows_out)
csv_df = df.copy()
for col in ['tracking_candidates', 'order_candidates']:
    if col in csv_df.columns:
        csv_df[col] = csv_df[col].apply(lambda v: '|'.join(v) if isinstance(v, list) else v)
csv_df.to_csv(RESULTS_CSV, index=False)
print(f"Saved → {RESULTS_CSV.resolve()}")

df[[
    'subject', 'sender_domain',
    'tracking_candidates', 'selected_tracking_number',
    'status', 'status_confidence', 'classification_method',
    'model_seconds', 'status_evidence',
]].head(30)

In [ ]:
print("=== Status distribution ===")
print(df['status'].value_counts().to_string())

print("\n=== Classification method ===")
print(df['classification_method'].value_counts().to_string())

unknown_model = df[(df['model_called'] == True) & (df['status'] == 'unknown')]
print(f"\nModel-unknown rows: {len(unknown_model)}")

long_inputs = df[df['cleaned_full_len'] > model_cfg.max_input_chars * 0.9]
print(f"Emails >90% of input cap ({model_cfg.max_input_chars} chars): {len(long_inputs)}")

---
## Part 4 — Persistence + shipment resolution

`ShipmentResolutionService` normalises identifiers, resolves canonical shipment
IDs via the alias table, inserts `shipment_events` rows (idempotent), and
rebuilds the `shipments` timeline row for every affected shipment.

In [ ]:
from parsli.db.repositories import EmailAccountRepository
from parsli.services.shipment_resolution_service import ShipmentResolutionService

# engine / session_factory already created in the DB setup cell at the top.
print(f"DB: {config.database.sqlite_path.resolve()}")

In [ ]:
from datetime import datetime, timezone
from parsli.processing.extraction_orchestrator import FinalExtraction
from parsli.domain.statuses import ShipmentStatus
from parsli.domain.identifiers import TrackingIdentifier, OrderIdentifier

_received_at = {
    e['id']: datetime.fromtimestamp(e['internal_date_ms'] / 1000, tz=timezone.utc)
    for e in emails
}

EMAIL_ACCOUNT_HASH = sha256_hex(ACCOUNT_ID)

with session_factory() as session:
    acct_repo = EmailAccountRepository(session)
    account   = acct_repo.find_by_hash(EMAIL_ACCOUNT_HASH)
    if account is None:
        account = acct_repo.create(EMAIL_ACCOUNT_HASH)

    resolution_svc = ShipmentResolutionService(session, config.processing)
    inserted = skipped = 0

    for row in rows_out:
        eid = row['email_id']
        extraction = FinalExtraction(
            email_id=eid,
            processing_version=config.processing.processing_version(),
            is_relevant=(
                row.get('is_shipping_email', False)
                and not row.get('is_invoice', False)
                and row.get('status', 'unknown') != 'unknown'
            ),
            ignore_reason=(
                'invoice' if row.get('is_invoice')
                else (None if row.get('is_shipping_email') else 'not_shipping_shaped')
            ),
            is_invoice=row.get('is_invoice', False),
            status=ShipmentStatus(row.get('status', 'unknown')),
            status_confidence=row.get('status_confidence', 0.0),
            status_evidence=row.get('status_evidence', ''),
            merchant=row.get('merchant'),
            selected_tracking_number=row.get('selected_tracking_number'),
            tracking_candidates=[TrackingIdentifier(value=v) for v in (row.get('tracking_candidates') or [])],
            selected_order_number=row.get('selected_order_number'),
            order_candidates=[OrderIdentifier(value=v) for v in (row.get('order_candidates') or [])],
            pickup_code=row.get('pickup_code'),
            amount=row.get('amount'),
            currency=row.get('currency'),
            model_provider=BACKEND if row.get('model_called') else None,
            model_name=model_cfg.model_name if row.get('model_called') else None,
            classification_method=row.get('classification_method', 'rules'),
        )
        resolution_svc.resolve_and_insert(extraction, _received_at.get(eid, datetime.now(timezone.utc)))
        (inserted if extraction.is_relevant else skipped).__class__  # just for the counter below
        if extraction.is_relevant:
            inserted += 1
        else:
            skipped += 1

    session.commit()
    print(f"Committed — {inserted} events inserted, {skipped} skipped (irrelevant/invoice)")

In [ ]:
from parsli.services.dashboard_service import DashboardService

with session_factory() as session:
    dash = DashboardService(session).get_dashboard()

print(f"Total: {dash.total_count}  Active: {dash.active_count}  Delivered: {dash.delivered_count}\n")

pd.DataFrame([{
    'id':           s.canonical_shipment_id,
    'merchant':     s.merchant,
    'tracking':     s.primary_tracking_number,
    'order':        s.primary_order_number,
    'status':       s.current_status.value,
    'status_date':  s.current_status_date.date(),
    'evidence':     s.current_status_evidence[:80],
    'events':       s.event_count,
    'chronology':   s.chronology_severity,
} for s in dash.shipments])

In [ ]:
# Chronology issues
issues = [s for s in dash.shipments if s.chronology_severity != 'ok']
print(f"Chronology issues: {len(issues)}")
for s in issues:
    print(f"\n  [{s.chronology_severity.upper()}] {s.canonical_shipment_id}")
    print(f"  tracking={s.primary_tracking_number}  merchant={s.merchant}")
    for note in s.chronology_notes:
        print(f"    • {note}")